In [1]:
import sys, platform, tempfile, pathlib

WORK = pathlib.Path(tempfile.mkdtemp(prefix="bfl_"))
print(f"{WORK.is_dir = }, {WORK.name.startswith('bfl_')}")

WORK.is_dir = <bound method Path.is_dir of WindowsPath('C:/Users/ramak/AppData/Local/Temp/bfl_gb3i85vb')>, True


In [2]:
import sqlite3
conn = sqlite3.connect(":memory:")               # a private database in RAM
cur = conn.cursor()
cur.execute('''
    CREATE TABLE loans (
        lan       TEXT PRIMARY KEY,
        customer  TEXT NOT NULL,
        branch    TEXT NOT NULL,
        principal INTEGER,
        dpd       INTEGER DEFAULT 0
    )''')

print("tables:", cur.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall())

tables: [('loans',)]


In [3]:
loans = [
    ("BFL001", "Ravi Kumar",     "Pune",   500000, 0),
    ("BFL002", "Anita D'Souza",  "Mumbai", 250000, 45),
    ("BFL003", "Sunil Patil",    "Pune",   800000, 12),
    ("BFL004", "Meera Joshi",    "Nagpur", 150000, 0),
    ("BFL005", "José Fernandes", "Mumbai", 1200000, 90),
]
cur.executemany("INSERT INTO loans VALUES (?, ?, ?, ?, ?)", loans)
conn.commit()
print("rows inserted:", cur.rowcount, "| total:", cur.execute("SELECT COUNT(*) FROM loans").fetchone()[0])

rows inserted: 5 | total: 5


In [4]:
# Querying the tables

cur.execute("SELECT lan, customer, dpd FROM loans WHERE dpd > ? ORDER BY dpd DESC", (0,))
print("fetchone:", cur.fetchone())
print("fetchall:", cur.fetchall())               # the rest
for lan, principal in conn.execute("SELECT lan, principal FROM loans WHERE branch = ? ORDER BY lan", ("Pune",)):
    print(f"  {lan}  ₹{principal:,}")

fetchone: ('BFL005', 'José Fernandes', 90)
fetchall: [('BFL002', "Anita D'Souza", 45), ('BFL003', 'Sunil Patil', 12)]
  BFL001  ₹500,000
  BFL003  ₹800,000


In [5]:
cur = conn.execute("UPDATE loans SET dpd = 0 WHERE lan = ?", ("BFL003",))
conn.commit()
print("rows updated:", cur.rowcount, "| BFL003 DPD now", conn.execute("SELECT dpd FROM loans WHERE lan = 'BFL003'").fetchone()[0])

rows updated: 1 | BFL003 DPD now 0


In [6]:
def dpd_of(lan):
    return conn.execute("SELECT dpd FROM loans WHERE lan = ?", (lan,)).fetchone()[0]

try:
    with conn:                                   # one transaction: all or nothing
        conn.execute("UPDATE loans SET dpd = 0 WHERE lan = ?", ("BFL005",))
        print("  inside the block, BFL005 DPD =", dpd_of("BFL005"))
        conn.execute("INSERT INTO loans VALUES (?, ?, ?, ?, ?)", ("BFL001", "Duplicate", "Pune", 1, 0))
except sqlite3.IntegrityError as e:
    print("IntegrityError:", e)
print("after rollback, BFL005 DPD =", dpd_of("BFL005"))
conn.close()
print("in-memory connection closed")

  inside the block, BFL005 DPD = 0
IntegrityError: UNIQUE constraint failed: loans.lan
after rollback, BFL005 DPD = 90
in-memory connection closed


In [7]:
# With db file

from contextlib import closing
db_path = WORK / "branch_book.db"
with closing(sqlite3.connect(db_path)) as db:    # closing(): close() at the end of the block
    with db:                                     # transaction: commit on success
        db.execute("CREATE TABLE IF NOT EXISTS emis (lan TEXT, month TEXT, amount REAL)")
        db.executemany("INSERT INTO emis VALUES (?, ?, ?)",
                       [("BFL001", "2026-11", 12500.5), ("BFL002", "2026-11", 8400.0)])

with closing(sqlite3.connect(db_path)) as db:    # a new connection: the data survived
    print(db.execute("SELECT COUNT(*), ROUND(SUM(amount), 2) FROM emis").fetchone())
print(db_path.name, "exists:", db_path.exists())

(2, 20900.5)
branch_book.db exists: True
